# Example Control: AIRBOT_Play机械臂控制

### 初始化机械臂

In [ ]:
from easy_arm_control.control import RoboticArmAgent, ChooseGripper
import rospy

# ROS节点初始化
NODE_NAME = "airbot_play_control_example"
rospy.init_node(NODE_NAME)
other_config = None
sim_player_nums = rospy.get_param("/airbot_play/sim_player_nums", default=0)
sim_type = ["real", "gazebo_i"][sim_player_nums]
print("Use", sim_type)
gripper = ["gripper", "inspire"][0]
gripper_controller = ChooseGripper(sim_type, gripper)()
arm = RoboticArmAgent(node_name=NODE_NAME, gripper=(4, gripper_controller))

### 机械臂控制

#### 关节空间控制

In [ ]:
import math

# 控制机械臂的关节0和5（从底座到夹爪关节依次编号0-6）转动到45度和90度绝对位置处，然后暂停2s
# 成功执行会看到返回True
arm.go_to_named_or_joint_target({0: math.pi / 4, 5: math.pi / 2}, sleep_time=2)

In [ ]:
import math

# 控制关节0和5相对当前位置转动-45度和-90度，然后暂停2s
arm.go_to_any_joint_target([0, 5], [-math.pi / 4, -math.pi / 2], sleep_time=2)

In [ ]:
# 控制所有关节转动到指定的角度位置，然后暂停2s
arm.go_to_named_or_joint_target([0, -0.3, 0.4, -0.3, 0, 0], sleep_time=2)

#### 工作空间控制

In [ ]:
# 控制机械臂末端到达指定的工作空间位置
position = [0.4, 0, 0.3]  # xyz位置值
rotation = [-3.14, 0, 0]  # rpy角度值或xyzw四元数
arm.set_and_go_to_pose_target(position, rotation, sleep_time=2)

In [ ]:
import math

# 控制机械臂末端增量式移动指定距离和角度
pos_inc = [0, 0, 0]  # xyz位置增量值
rot_inc = [math.pi / 2, math.pi / 10, math.pi / 10]  # rpy角度增量值
arm.set_and_go_to_pose_target(
    pos_inc, rot_inc, "last", sleep_time=2
)  # last表示给定值是基于上次目标值的增量

In [ ]:
# 单轴绝对式移动：控制机械臂末端沿x轴移动到0.08m处，其它轴位置保持不变
arm.go_to_single_axis_target(0, 0.28, sleep_time=2)

In [ ]:
# 增量式移动：控制机械臂末端沿x轴正方向移动0.05m，其它轴位置保持不变
arm.go_to_shift_single_axis_target(0, 0.05, sleep_time=2)

In [ ]:
# 查询目标位置是否可达
arm.check_pose((0, 1, 2, 3, 4, 5))

#### 末端执行器控制

In [ ]:
# 1：夹爪闭合，0：夹爪张开
arm.gripper_control(1, sleep_time=1.5)

In [ ]:
# 1：夹爪闭合，0：夹爪张开
arm.gripper_control(0, sleep_time=1.5)

#### 信息获取

In [ ]:
# 获取关节状态
current_joint_states = arm.get_current_joint_values()
# 获取末端位姿
current_pose = arm.get_current_pose()
# 终端打印各种信息
arm.log_arm_info(
    current_joint=True,
    current_pose=True,
    target_joint=True,
    target_pose=True,
    raw_info=True,
)

#### 退出ROS节点

In [ ]:
# 关闭当前控制节点
arm.go_to_named_or_joint_target("Home", sleep_time=3)
rospy.signal_shutdown("end")